In [1]:
import xarray as xr
import numpy as np

In [4]:
atm_3d = xr.open_dataset("/wolke_scratch/dnikolo/ICON_Data/cosp_dardar_runs/R2B4/wbf_v3/raw_data/sedim_r2b4_wbfv3_atm_3d_cloud_ml_20071221T000000Z.nc")
ct = xr.open_dataset("/wolke_scratch/dnikolo/ICON_Data/cosp_dardar_runs/R2B4/wbf_v3/raw_data/sedim_r2b4_wbfv3_ctp_2d_ml_20071221T000000Z.nc")

In [4]:
ct["ctmask"]

<xarray.DataArray 'ctmask' (time: 96, ncells: 20480)>
[1966080 values with dtype=float32]
Coordinates:
    clon     (ncells) float32 ...
    clat     (ncells) float32 ...
  * time     (time) float64 2.006e+07 2.006e+07 ... 2.006e+07 2.006e+07
Dimensions without coordinates: ncells
Attributes:
    standard_name:                ctmask
    long_name:                    Cloud top mask
    units:                        -
    param:                        79.1.0
    CDI_grid_type:                unstructured
    number_of_grid_in_reference:  1
    level_type:                   cloudtop

In [5]:
import numpy as np
import xarray as xr

ctmask_3d = (
    ct["ctmask"]
    .expand_dims(height=atm_3d.height)
    .broadcast_like(atm_3d["cli"])
    .transpose("time", "height", "ncells")
)
CPH = xr.where(((ctmask_3d>1e-5) & ((atm_3d["cli"]+atm_3d["clw"])>=1e-5)),(atm_3d["cli"]/(atm_3d["cli"]+atm_3d["clw"])+1),0)

# ctlev is probably 1-based because height is 1..47
# Convert model level number to zero-based positional index
ctlev_idx = ct["ctlev"].astype(float)

rel_level = xr.DataArray(
    np.arange(4),
    dims="rel_level",
    coords={"rel_level": np.arange(4)}
)

# cloud top + 1/2/3 levels below
target_idx = ctlev_idx + rel_level
nlev = atm_3d.sizes["height"]

valid = (
    np.isfinite(target_idx) &
    (target_idx >= 0) &
    (target_idx < nlev)
)

target_idx_safe = (
    target_idx
    .where(valid, 0)
    .astype(int)
)

CPH_ct_profile = (
    CPH
    .isel(height=target_idx_safe)
    .where(valid, other=0)
    .transpose("time", "rel_level", "ncells")
)

In [6]:
atm_3d

<xarray.Dataset>
Dimensions:           (ncells: 20480, vertices: 3, height: 47, bnds: 2, time: 96)
Coordinates:
    clon              (ncells) float32 ...
    clat              (ncells) float32 ...
  * height            (height) float64 1.0 2.0 3.0 4.0 ... 44.0 45.0 46.0 47.0
  * time              (time) float64 2.007e+07 2.007e+07 ... 2.007e+07 2.007e+07
Dimensions without coordinates: ncells, vertices, bnds
Data variables:
    clon_bnds         (ncells, vertices) float32 ...
    clat_bnds         (ncells, vertices) float32 ...
    height_bnds       (height, bnds) float64 ...
    ps                (time, ncells) float32 ...
    pfull             (time, height, ncells) float32 ...
    ta                (time, height, ncells) float32 ...
    clw               (time, height, ncells) float32 ...
    cli               (time, height, ncells) float32 ...
    cl                (time, height, ncells) float32 ...
    cloudsat_ze_mean  (time, height, ncells) float32 ...
    zg                (height, ncells) float32 ...
Attributes:
    CDI:                  Climate Data Interface version 1.8.3rc (http://mpim...
    Conventions:          CF-1.6
    number_of_grid_used:  5
    grid_file_uri:        http://icon-downloads.mpimet.mpg.de/grids/public/ic...
    uuidOfHGrid:          6dffb27e-c162-11e6-bde4-f3bd241f4de0
    uuidOfVGrid:          75ae1dba-8996-102c-7d45-dfaaabcd2a80
    institution:          Max Planck Institute for Meteorology/Deutscher Wett...
    title:                ICON simulation
    source:               git@git.iac.ethz.ch:hammoz/icon-for-hammoz.git@f452...
    history:              /capstor/store/cscs/userlab/lp88/dnikolov/icon-for-...
    references:           see MPIM/DWD publications
    comment:              Dragomir Nikolov, ETHZ (dnikolov) on nid005119 (Lin...

In [13]:
atm_3d["zg"].isel(ncells=1 ).values

array([80046.695  , 74659.77   , 70018.84   , 65768.69   , 61816.67   ,
       58112.832  , 54624.633  , 51328.7    , 48207.125  , 45245.613  ,
       42432.375  , 39757.484  , 37212.4    , 34789.688  , 32482.803  ,
       30285.912  , 28193.783  , 26201.697  , 24305.365  , 22500.87   ,
       20784.627  , 19153.338  , 17603.957  , 16134.173  , 14741.227  ,
       13422.394  , 12175.843  , 10999.651  ,  9892.071  ,  8851.512  ,
        7876.5186 ,  6965.7456 ,  6117.9287 ,  5331.85   ,  4606.306  ,
        3940.0842 ,  3331.9417 ,  2780.6082 ,  2284.8052 ,  1843.2899 ,
        1454.9222 ,  1118.755  ,   834.149  ,   600.92194,   419.5749 ,
         292.2258 ,   222.46617], dtype=float32)

In [14]:
heights = atm_3d["zg"].isel(ncells=1 ).values
(heights[:-1]- heights[1:])/1000

array([5.386922  , 4.6409297 , 4.2501564 , 3.9520156 , 3.7038398 ,
       3.4881992 , 3.2959335 , 3.1215742 , 2.9615116 , 2.8132384 ,
       2.6748905 , 2.545086  , 2.422711  , 2.3068848 , 2.1968906 ,
       2.092129  , 1.9920859 , 1.896332  , 1.804496  , 1.7162422 ,
       1.631289  , 1.5493809 , 1.4697841 , 1.3929462 , 1.318833  ,
       1.2465508 , 1.1761914 , 1.1075801 , 1.0405595 , 0.97499317,
       0.9107729 , 0.8478169 , 0.78607863, 0.7255439 , 0.6662219 ,
       0.60814255, 0.5513335 , 0.49580297, 0.44151527, 0.38836768,
       0.33616725, 0.284606  , 0.23322704, 0.18134704, 0.1273491 ,
       0.06975963], dtype=float32)

In [ ]:
for i in range(len(heights)):
    print(np.sum(heights))

In [12]:
atm_3d["zg"].isel(height=20,ncells=1 ).values

array(20784.627, dtype=float32)

In [15]:
import numpy as np
import xarray as xr


# ============================================================
# User settings
# ============================================================

max_depth_m = 2160.0
dz_m = 120.0

# Change these if your variable names are different
zg_name = "zg"
ze_name = "cloudsat_ze_mean"
iwc_name = "cli"

# If ctlev ranges from 1..47, set this to True.
# If ctlev ranges from 0..46, set this to False.
ctlev_is_one_based = False


# ============================================================
# Target cloud-top-relative vertical grid
# ============================================================

# Bin centers: 60, 180, ..., 2100 m
depth_bins = xr.DataArray(
    np.arange(dz_m / 2, max_depth_m, dz_m),
    dims="depth_bin",
    coords={"depth_bin": np.arange(dz_m / 2, max_depth_m, dz_m)},
    name="depth_below_ct_m",
)

depth_bins.attrs["long_name"] = "Depth below cloud top"
depth_bins.attrs["units"] = "m"


# ============================================================
# Cloud-top level handling
# ============================================================

nlev = atm_3d.sizes["height"]

ctlev_raw = ct["ctlev"].astype(float)

if ctlev_is_one_based:
    ctlev_idx = ctlev_raw - 1
else:
    ctlev_idx = ctlev_raw

valid_ctlev = (
    np.isfinite(ctlev_idx) &
    (ctlev_idx >= 0) &
    (ctlev_idx < nlev) &
    (ct["ctmask"] > 1e-5)
)

ctlev_idx_safe = (
    ctlev_idx
    .where(valid_ctlev, 0)
    .astype(int)
)


# ============================================================
# Get cloud-top geometric height
# ============================================================

zg = (
    atm_3d[zg_name]
    .broadcast_like(atm_3d["cli"])
    .transpose("time", "height", "ncells")
)

zg_ct = zg.isel(height=ctlev_idx_safe)

# Positive depth means below cloud top
depth_below_ct = zg_ct - zg

depth_below_ct = depth_below_ct.transpose("time", "height", "ncells")
depth_below_ct.name = "depth_below_ct"


# ============================================================
# Interpolation helper
# ============================================================

def interp_to_depth_bins(depth_col, var_col, target_depths):
    """
    Interpolate one vertical column to fixed cloud-top-relative depths.

    Parameters
    ----------
    depth_col : array, shape (height,)
        Depth below cloud top in meters. Positive values are below cloud top.

    var_col : array, shape (height,)
        Variable profile to interpolate.

    target_depths : array, shape (depth_bin,)
        Target depths below cloud top.

    Returns
    -------
    out : array, shape (depth_bin,)
        Interpolated profile.
    """

    depth_col = np.asarray(depth_col)
    var_col = np.asarray(var_col)
    target_depths = np.asarray(target_depths)

    valid = (
        np.isfinite(depth_col) &
        np.isfinite(var_col) &
        (depth_col >= 0.0) &
        (depth_col <= max_depth_m)
    )

    if valid.sum() < 2:
        return np.full(target_depths.shape, np.nan, dtype=float)

    x = depth_col[valid]
    y = var_col[valid]

    order = np.argsort(x)
    x = x[order]
    y = y[order]

    # Remove duplicate depth values
    x_unique, unique_idx = np.unique(x, return_index=True)
    y_unique = y[unique_idx]

    if len(x_unique) < 2:
        return np.full(target_depths.shape, np.nan, dtype=float)

    return np.interp(
        target_depths,
        x_unique,
        y_unique,
        left=np.nan,
        right=np.nan,
    )


# ============================================================
# Generic function for one variable
# ============================================================

def make_ct_relative_profile(var):
    """
    Convert a 3D variable with dims (time, height, ncells)
    into cloud-top-relative depth coordinates.

    Output dims: (time, depth_bin, ncells)
    """

    var = var.transpose("time", "height", "ncells")

    out = xr.apply_ufunc(
        interp_to_depth_bins,
        depth_below_ct,
        var,
        depth_bins,
        input_core_dims=[["height"], ["height"], ["depth_bin"]],
        output_core_dims=[["depth_bin"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
        dask_gufunc_kwargs={"allow_rechunk": True},
    )

    out = out.transpose("time", "depth_bin", "ncells")

    # Mask invalid cloud-top pixels
    out = out.where(valid_ctlev)

    return out


# ============================================================
# Prepare variables
# ============================================================

ze = atm_3d[ze_name]
iwc = atm_3d[iwc_name]


# ============================================================
# Option A: interpolate Ze directly
# ============================================================

ze_depth = make_ct_relative_profile(ze)
iwc_depth = make_ct_relative_profile(iwc)


# ============================================================
# Option B: better for reflectivity statistics
# Convert dBZ -> linear Z, interpolate, then back to dBZ.
# Use this if ze is in dBZ.
# ============================================================

# Z_linear = 10.0 ** (ze / 10.0)
# Z_linear_depth = make_ct_relative_profile(Z_linear)
# ze_depth = 10.0 * np.log10(Z_linear_depth)


# ============================================================
# Optional: also compute CPH and interpolate it
# ============================================================

ctmask_3d = (
    ct["ctmask"]
    .expand_dims(height=atm_3d.height)
    .broadcast_like(atm_3d["cli"])
    .transpose("time", "height", "ncells")
)

condensate = atm_3d["cli"] + atm_3d["clw"]

CPH = xr.where(
    ((ctmask_3d > 1e-5) & (condensate >= 1e-5)),
    atm_3d["cli"] / condensate + 1.0,
    0.0,
)

CPH_depth = make_ct_relative_profile(CPH)


# ============================================================
# Final compact dataset
# ============================================================

ct_relative_ds = xr.Dataset(
    data_vars={
        "ze": ze_depth,
        "iwc": iwc_depth,
        "cph": CPH_depth,
    },
    coords={
        "time": atm_3d["time"],
        "depth_bin": depth_bins,
        "ncells": atm_3d["ncells"],
        "clon": atm_3d["clon"],
        "clat": atm_3d["clat"],
    },
)

ct_relative_ds["ze"].attrs = atm_3d[ze_name].attrs
ct_relative_ds["iwc"].attrs = atm_3d[iwc_name].attrs
ct_relative_ds["cph"].attrs["long_name"] = "Cloud phase interpolated to cloud-top-relative depth"
ct_relative_ds["cph"].attrs["units"] = "1 liquid, 2 ice"

ct_relative_ds["depth_bin"].attrs["long_name"] = "Depth below cloud top"
ct_relative_ds["depth_bin"].attrs["units"] = "m"


# ============================================================
# Check result
# ============================================================

print(ct_relative_ds)

print("ze shape:", ct_relative_ds["ze"].shape)
print("iwc shape:", ct_relative_ds["iwc"].shape)
print("cph shape:", ct_relative_ds["cph"].shape)

<xarray.Dataset>
Dimensions:    (ncells: 20480, time: 96, depth_bin: 18)
Coordinates:
    clon       (ncells) float32 0.6283 0.6283 0.5973 ... 0.2479 0.2626 0.2156
    clat       (ncells) float32 0.9184 0.9405 0.9072 ... -0.7248 -0.6888 -0.6939
  * time       (time) float64 2.007e+07 2.007e+07 ... 2.007e+07 2.007e+07
  * depth_bin  (depth_bin) float64 60.0 180.0 300.0 ... 1.98e+03 2.1e+03
  * ncells     (ncells) int64 0 1 2 3 4 5 ... 20475 20476 20477 20478 20479
Data variables:
    ze         (time, depth_bin, ncells) float64 nan nan nan nan ... nan nan nan
    iwc        (time, depth_bin, ncells) float64 nan nan nan nan ... nan nan nan
    cph        (time, depth_bin, ncells) float64 nan nan nan nan ... nan nan nan
ze shape: (96, 18, 20480)
iwc shape: (96, 18, 20480)
cph shape: (96, 18, 20480)


In [33]:
np.isnan(atm_3d["cloudsat_ze_mean"]).mean()

<xarray.DataArray 'cloudsat_ze_mean' ()>
array(0.42317547)

In [21]:
np.isnan(ct_relative_ds["ze"]).all()

<xarray.DataArray 'ze' ()>
array(True)

In [21]:
(ct["ctmask"]).mean()

<xarray.DataArray 'ctmask' ()>
array(0.73106724, dtype=float32)

In [23]:
(CPH_ct_profile.sel(rel_level=0)>=1).mean()

<xarray.DataArray ()>
array(0.6219101)
Coordinates:
    rel_level  int64 0